In [1]:
%pip -q install -U sentence-transformers gradio beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 3.6 MB/s eta 0:00:00


In [2]:
import os
import re
import html
import numpy as np
import pandas as pd
import torch
import gradio as gr

from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Load the dataset directly from GitHub
DATASET_URL = "https://raw.githubusercontent.com/shyamgayke/Deja_Code/main/questions.csv"

df = pd.read_csv(DATASET_URL)

# Check required columns
required_columns = ["QCode", "Title", "level", "link", "statement"]
missing = set(required_columns) - set(df.columns)
assert not missing, f"Missing columns: {missing}"

# Display dataset information
print("Dataset shape:", df.shape)
print("Number of problems:", len(df))
display(df[required_columns].head())


Dataset shape: (1474, 13)
Number of problems: 1474


,QCode,Title,level,link,statement
0,SEQUENCE,Integer Sequences,medium,/problems/SEQUENCE,\nAll submissions for this problem are availab...
1,CHEFTEAM,Chef team,easy,/problems/CHEFTEAM,\nAll submissions for this problem are availab...
2,SEATR,Sereja and Tree,medium,/problems/SEATR,\nAll submissions for this problem are availab...
3,FLOW017,Second Largest,beginner,/problems/FLOW017,\nAll submissions for this problem are availab...
4,CFRTEST,Devu and friendship testing,beginner,/problems/CFRTEST,\nAll submissions for this problem are availab...


In [3]:
def clean_html(text):
    # Convert HTML entities and remove HTML tags
    text = html.unescape(str(text))
    soup = BeautifulSoup(text, "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    return re.sub(r"\s+", " ", text).strip()


df_clean = df.copy()

df_clean["clean_statement"] = (
    df_clean["statement"].fillna("").apply(clean_html)
)

# Combine titles and statements for text retrieval
df_clean["search_text"] = (
    df_clean["Title"].fillna("").astype(str)
    + ". "
    + df_clean["clean_statement"]
)

print("Problems processed:", len(df_clean))
print(
    "Empty statements:",
    (df_clean["clean_statement"].str.len() == 0).sum()
)

display(df_clean[["QCode", "Title", "search_text"]].head(3))


Problems processed: 1474
Empty statements: 0


,QCode,Title,search_text
0,SEQUENCE,Integer Sequences,Integer Sequences. All submissions for this pr...
1,CHEFTEAM,Chef team,Chef team. All submissions for this problem ar...
2,SEATR,Sereja and Tree,Sereja and Tree. All submissions for this prob...


In [4]:
# Convert problem text into keyword-based vectors
vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(df_clean["search_text"])


def search_tfidf(query, top_k=5):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(
        query_vector, tfidf_matrix
    ).flatten()

    top_indices = np.argsort(-scores)[:top_k]

    results = df_clean.iloc[top_indices][
        ["QCode", "Title", "level", "link"]
    ].copy()

    results["similarity"] = scores[top_indices]
    return results.reset_index(drop=True)


print("TF-IDF index shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))


TF-IDF index shape: (1474, 50000)
Vocabulary size: 50000


In [5]:
# Load a pretrained sentence embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)

# Embed each title and the first 1,200 characters of its statement
semantic_texts = (
    df_clean["Title"].fillna("").astype(str)
    + ". "
    + df_clean["clean_statement"].fillna("").str.slice(0, 1200)
).tolist()

embeddings = model.encode(
    semantic_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)
print("Device:", device)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

Embedding shape: (1474, 384)
Device: cuda


In [6]:
def search_semantic(query, top_k=5):
    query = query.strip()

    if not query:
        return pd.DataFrame(
            columns=["QCode", "Title", "level", "link", "similarity"]
        )

    # Embed the user's query
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    # Compare the query against all problem embeddings
    scores = embeddings @ query_embedding
    top_indices = np.argsort(-scores)[:top_k]

    results = df_clean.iloc[top_indices][
        ["QCode", "Title", "level", "link"]
    ].copy()

    results["similarity"] = scores[top_indices]
    return results.reset_index(drop=True)


# Quick smoke test
display(
    search_semantic(
        "find the maximum sum of a contiguous subarray",
        top_k=5
    )
)


,QCode,Title,level,link,similarity
0,MMSUM,Better Maximal Sum,easy,/problems/MMSUM,0.718281
1,MXSUBARR,Maximum Disjoint Subarrays,easy,/problems/MXSUBARR,0.647891
2,PAIRSUM,Sum of pairs,easy,/problems/PAIRSUM,0.567006
3,SUBLCM,Subarray LCM,medium,/problems/SUBLCM,0.566142
4,MAXRECT,Maximum Sub-rectangle in Matrix,challenge,/problems/MAXRECT,0.559751


In [7]:
# Manually judged relevant codes for development testing
evaluation_set = [
    {
        "query": "count the number of ways to choose K people from N",
        "relevant": {"CHEFTEAM"}
    },
    {
        "query": "find the shortest path in a graph",
        "relevant": {"TACTQUER", "SPSHORT", "AVGSHORT"}
    },
    {
        "query": "longest increasing subsequence",
        "relevant": {"C5", "MAKELIS", "LISLDS"}
    },
    {
        "query": "determine whether a number is prime",
        "relevant": {"PRB01"}
    },
    {
        "query": "find the maximum sum of a contiguous subarray",
        "relevant": {"MMSUM"}
    }
]


def evaluate_retriever(search_function, cases, k=5):
    precision_scores = []
    recall_scores = []
    reciprocal_ranks = []

    for case in cases:
        results = search_function(case["query"], top_k=k)
        retrieved = results["QCode"].tolist()
        relevant = case["relevant"]

        hits = sum(code in relevant for code in retrieved)

        precision_scores.append(hits / k)
        recall_scores.append(hits / len(relevant))

        first_rank = next(
            (
                rank for rank, code in enumerate(retrieved, start=1)
                if code in relevant
            ),
            None
        )

        reciprocal_ranks.append(
            1 / first_rank if first_rank is not None else 0
        )

    return {
        f"Precision@{k}": np.mean(precision_scores),
        f"Recall@{k}": np.mean(recall_scores),
        "MRR": np.mean(reciprocal_ranks)
    }


comparison = pd.DataFrame([
    {
        "Method": "TF-IDF",
        **evaluate_retriever(search_tfidf, evaluation_set)
    },
    {
        "Method": "Semantic",
        **evaluate_retriever(search_semantic, evaluation_set)
    }
])

display(comparison.round(3))


,Method,Precision@5,Recall@5,MRR
0,TF-IDF,0.28,0.733,0.7
1,Semantic,0.36,1.000,1.0


In [8]:
def deja_code_search(query, difficulty, top_k):
    columns = [
        "QCode", "Title", "Difficulty",
        "Semantic Similarity", "Problem Link"
    ]

    if not query or not query.strip():
        return pd.DataFrame(columns=columns)

    results = search_semantic(
        query.strip(),
        top_k=int(top_k)
    )

    # Filter results by difficulty
    if difficulty != "All":
        results = results[
            results["level"].str.strip().str.lower()
            == difficulty.strip().lower()
        ]

    # Format the results for display
    output = pd.DataFrame({
        "QCode": results["QCode"],
        "Title": results["Title"],
        "Difficulty": results["level"],
        "Semantic Similarity": results["similarity"].round(3),
        "Problem Link": results["link"].apply(
            lambda path: (
                str(path) if str(path).startswith("http")
                else "https://www.codechef.com" + str(path)
            )
        )
    })

    return output[columns].reset_index(drop=True)


with gr.Blocks(title="Déjà Code") as demo:
    gr.Markdown("# Déjà Code")
    gr.Markdown(
        "Find similar CodeChef problems using natural language. "
        "Describe a problem, algorithm, or programming concept."
    )

    query_input = gr.Textbox(
        label="Describe the problem",
        placeholder="e.g. Find the shortest path in a weighted graph",
        lines=2
    )

    with gr.Row():
        difficulty_input = gr.Dropdown(
            choices=[
                "All", "Beginner", "Easy",
                "Medium", "Hard", "Challenge"
            ],
            value="All",
            label="Difficulty"
        )

        top_k_input = gr.Slider(
            minimum=3,
            maximum=10,
            value=5,
            step=1,
            label="Number of results"
        )

    search_button = gr.Button("Search", variant="primary")

    results_output = gr.Dataframe(
        label="Similar problems",
        interactive=False
    )

    search_button.click(
        fn=deja_code_search,
        inputs=[query_input, difficulty_input, top_k_input],
        outputs=results_output
    )

    query_input.submit(
        fn=deja_code_search,
        inputs=[query_input, difficulty_input, top_k_input],
        outputs=results_output
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2b6ffe524103445073.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
